# HHL Master Notebook: From the Original QComp Session to a Solved Research Walkthrough

This notebook is the central notebook of the repository. It is adapted from the original QComp HHL session, but reworked into a fully solved, portfolio-ready walkthrough aligned with the codebase.

It covers:

1. statevector conventions and basis ordering
2. the fixed `2x2` Hermitian system used by the project
3. the exact eigenphase encoding that makes the example clean
4. the inversion oracle and postselection mechanism
5. the complete HHL circuit and recovered solution
6. precision and parameter studies
7. an extension to a simple signed-eigenvalue case


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import numpy as np
from scipy.linalg import expm
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.quantum_info import Statevector

from hhl_lab.analysis import precision_sweep, summarize_spectrum
from hhl_lab.hhl import build_hhl_circuit
from hhl_lab.inversion import append_basis_controlled_ry
from hhl_lab.matrices import classical_solution, default_problem, normalized_classical_solution
from hhl_lab.qpe import append_qpe
from hhl_lab.simulation import run_hhl_statevector
from hhl_lab.visualization import (
    plot_eigenvalue_encoding,
    plot_precision_sweep,
    plot_solution_comparison,
    plot_spectral_decomposition,
)

np.set_printoptions(precision=6, suppress=True)


## 0. Background: reading Qiskit statevectors

One subtlety that matters throughout HHL is basis ordering. In Qiskit statevectors, the leftmost ket symbol corresponds to the highest-index qubit in little-endian order. The short example below makes that visible before we move on to HHL itself.

In [ ]:
q1 = QuantumRegister(3, name='q1')
q2 = QuantumRegister(1, name='q2')
toy = QuantumCircuit(q1, q2)
toy.x(q1[0])
toy.h(q1[1])
toy.h(q2[0])
toy.cx(q2[0], q1[2])
toy.draw('text')


In [ ]:
toy_sv = Statevector.from_instruction(toy)
for index, amplitude in enumerate(toy_sv.data):
    if abs(amplitude) > 1e-9:
        print(format(index, '04b'), amplitude)


## 1. Case Study Used Throughout the Repository

The central example is

$$
A = \begin{bmatrix} 1 & -1/3 \\ -1/3 & 1 \end{bmatrix},
\qquad
b = \begin{bmatrix} 1 \\ 0 \end{bmatrix},
\qquad
t = 3\pi/4.
$$

The classical solution is known exactly, and the quantum experiment is arranged so that the phase register can encode the eigenphases exactly with only two qubits.

In [ ]:
problem = default_problem()
print('A =')
print(problem.matrix)
print('\nb =')
print(problem.rhs)
print('\nt =', problem.evolution_time)
print('\nClassical solution A^-1 b =')
print(classical_solution(problem))
print('\nNormalized classical solution =')
print(normalized_classical_solution(problem))


## 2. Spectral Decomposition and Exact Phase Encoding

HHL acts in the eigenbasis of the Hermitian matrix. For this problem, both eigenvalues and eigenvectors are simple enough to inspect directly.

In [ ]:
summary = summarize_spectrum(problem)
print('Eigenvalues:')
print(summary.eigenvalues)
print('\nEigenvectors (columns):')
print(summary.eigenvectors)
print('\nCoefficients of |b> in the eigenbasis:')
print(summary.eigenbasis_coefficients)
print('\nPhase fractions lambda_j * t / (2*pi):')
print(summary.eigenvalues * problem.evolution_time / (2.0 * np.pi))


In [ ]:
plot_spectral_decomposition(run_hhl_statevector(problem))


The two phase fractions are `1/4` and `1/2`. That is the decisive reason why a two-qubit clock register is sufficient here: there is no rounding error at the QPE stage.

## 3. The Hamiltonian Simulation Block and QPE

The HHL construction uses the unitary $U = e^{iAt}$. The cell below reconstructs a two-qubit QPE stage around that unitary and verifies which computational basis states appear for one eigenstate.

In [ ]:
print('U = exp(i A t)')
print(expm(1j * problem.matrix * problem.evolution_time))


In [ ]:
system = QuantumRegister(1, 'sys')
clock = QuantumRegister(2, 'clk')
qpe_demo = QuantumCircuit(system, clock)
qpe_demo.h(system[0])
append_qpe(qpe_demo, [1, 2], 0, problem)
qpe_demo.draw('text')


In [ ]:
qpe_sv = Statevector.from_instruction(qpe_demo)
for index, amplitude in enumerate(qpe_sv.data):
    if abs(amplitude) > 1e-9:
        print(format(index, '03b'), amplitude)


## 4. Solving the Inversion Subroutine Explicitly

The original QComp session asked for a basis-controlled inversion oracle built from controlled $R_y$ rotations. The next cell reproduces that idea directly on the two exact clock states used by the project.

In [ ]:
qe = QuantumRegister(2, 'eig')
qi = QuantumRegister(1, 'inv')
qcinv = QuantumCircuit(qe, qi)

# With t = 3*pi/4 and C = 1/4 in the lecture-note convention, the effective
# reciprocal amplitudes are 1 and 1/2 for the two eigenvalue branches.
append_basis_controlled_ry(qcinv, [0, 1], 2, '10', np.pi)
append_basis_controlled_ry(qcinv, [0, 1], 2, '01', 2.0 * np.arcsin(0.5))
qcinv.draw('text')


This is exactly the step that injects the inverse eigenvalue dependence into the ancilla amplitude. After postselection, those weights become the reason why the final system amplitudes are proportional to the classical solution.

## 5. Complete HHL Circuit and Postselected Solution Recovery

We now run the full project implementation and inspect the recovered amplitudes.

In [ ]:
result = run_hhl_statevector(problem)
print('Raw postselected amplitudes:')
print(result.raw_solution_amplitudes)
print('\nNormalized HHL solution:')
print(result.normalized_hhl_solution)
print('\nState fidelity with normalized classical solution:')
print(result.fidelity_with_classical)
print('\nRelative L2 error:')
print(result.relative_error)
print('\nPostselection success probability:')
print(result.success_probability)


In [ ]:
plot_solution_comparison(result)


In [ ]:
plot_eigenvalue_encoding(result)


In [ ]:
artifacts = build_hhl_circuit(problem)
artifacts.circuit.draw('text')


The branch selected by the project is the one where the inversion ancilla is in `|1>` and the phase register has been uncomputed back to `|00>`. The extracted amplitudes are `[3/4, 1/4]`, which are proportional to the classical vector `[9/8, 3/8]`.

## 6. Precision and Parameter Study

The original notebook left a sequence of explorations about precision, `C`, and `t`. In the project version, these are summarized by a reproducible sweep over phase-register precision.

In [ ]:
points = precision_sweep(range(1, 7), problem)
for point in points:
    print(
        point.precision_bits,
        point.phase_bitstrings,
        point.encoded_eigenvalues,
        point.fidelity_with_classical,
        point.l2_error,
        point.is_exact_encoding,
    )


In [ ]:
plot_precision_sweep(points)


Interpretation:

- with one phase qubit, the smaller eigenphase aliases to zero, so inversion becomes ill-defined
- with two phase qubits, the encoding is already exact
- adding more qubits does not improve this particular example because it already sits exactly on the binary grid


## 7. Extending the Discussion: a Signed-Eigenvalue Example

The original QComp notebook also asked how to handle negative eigenvalues. A compact example is

$$
A = \begin{bmatrix}0 & 1 \\ 1 & 0\end{bmatrix},
\qquad
b = \frac{1}{\sqrt2}\begin{bmatrix}1 \\ -1\end{bmatrix},
$$

for which $b$ is already an eigenvector associated with eigenvalue $-1$. The classical solution is therefore simply $x = -b$.

In [ ]:
A_signed = np.array([[0.0, 1.0], [1.0, 0.0]])
b_signed = np.array([1.0, -1.0]) / np.sqrt(2.0)
x_signed = np.linalg.solve(A_signed, b_signed)
eigvals_signed, eigvecs_signed = np.linalg.eigh(A_signed)
coeffs_signed = eigvecs_signed.conj().T @ b_signed
print('Eigenvalues:', eigvals_signed)
print('Classical solution:', x_signed)
print('Eigenbasis coefficients:', coeffs_signed)


To handle negative eigenvalues, the interpretation of the phase register must shift from `[0, 1)` to `[-1/2, 1/2)`. In other words, bitstrings above `1/2` must be read as negative phases. The full package does not yet promote this signed-phase extension to the main API, but the notebook records the mathematical fix that resolves the original exercise.

In [ ]:
def signed_fraction_from_bitstring(bitstring):
    raw = int(bitstring, 2) / (2 ** len(bitstring))
    return raw if raw < 0.5 else raw - 1.0

test_bitstrings = ['00000', '01000', '10000', '11000']
[(bits, signed_fraction_from_bitstring(bits)) for bits in test_bitstrings]


That signed-phase reinterpretation is the essential repair: once the encoded phase is read in `[-1/2, 1/2)`, the reciprocal rotation can treat negative eigenvalues correctly as well.

## Conclusion

This notebook now serves as the main notebook of the project. It preserves the spirit of the original QComp session while removing the unfinished homework scaffolding, filling the missing derivations, and reconnecting every stage to the reusable project package.